In [20]:
import xtrack as xt
from helpers_for_imperfections_model import generate_monitor_misalignments, apply_monitor_misalignments, create_elements_switch, apply_orbit_correction
from helpers_from_xutil import match_tune_chroma

In [21]:
# CHOOSE SEED HERE
seed = 4
# CHOOSE LINE VERSION HERE
line_version = "LCC_105-0-0_z"

In [22]:
# Load reference line
line0 = xt.Line.from_json(f'line_fccee_p_ring_{line_version}.json')
line0.config.XTRACK_USE_EXACT_DRIFTS = False
line0.cycle(name_first_element='rf400', inplace=True) # cycle to rf cavity
line0.configure_radiation(model=None, model_beamstrahlung=None) # disable radiation 
line0.twiss_default['method'] = '4d' # switch to 4d Twiss method
print(f'Loaded the {line_version} reference line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.')

tt = line0.get_table()
tw_ref = line0.twiss() #tw_ref has 4D Twiss

Loading line from dict: 100%|██████████| 21288/21288 [00:02<00:00, 7606.53it/s] 


Done loading line from dict.           
Loaded the LCC_105-0-0_z reference line, cycled it to the RF cavity, disabled radiation, and switched to 4D Twiss.


In [23]:
# Load line with imperfections switches installed
line = xt.Line.from_json(f'{line_version}_line_with_imperfections_switches_seed{seed}.json')
# Remember that the line is cycled to rf cavity, has radiation disabled, and has Twiss method set to 4D

Loading line from dict: 100%|██████████| 21288/21288 [00:02<00:00, 7290.82it/s] 


Done loading line from dict.           


### Preparing the line for the correction procedure

In [24]:
# Generate the monitor misalignment dictionary

# Remember to set all the girder and quadrupole misalignment switches to 1 before calling the function
# Since bpms inherit their misalignments!
line.vars['on_misalignment_quad_arc'] = 1
line.vars['on_misalignment_quad_fd'] = 1
line.vars['on_misalignment_quad_fd_cs'] = 1
line.vars['on_misalignment_quad_cs_arc'] = 1
line.vars['on_misalignment_girder'] = 1
monitor_alignment = generate_monitor_misalignments(line, pattern='bpm', attrs=['shift_x', 'shift_y', 'rot_s_rad'], 
                                                    line_table=tt, element_type='Marker')
line.vars['on_misalignment_quad_arc'] = 0
line.vars['on_misalignment_quad_fd'] = 0
line.vars['on_misalignment_quad_fd_cs'] = 0
line.vars['on_misalignment_quad_cs_arc'] = 0
line.vars['on_misalignment_girder'] = 0

In [25]:
# Apply the additional bpm misalignments
apply_monitor_misalignments(monitor_alignment, seed, sigmas=[10e-6, 10e-6, 10e-6])

{'bpm_qd12f': {'shift_x': np.float64(7.916584534684518e-05),
  'shift_y': np.float64(1.0517270372780401e-05),
  'rot_s_rad': np.float64(2.1210669299018018e-05)},
 'bpm_qf13f': {'shift_x': np.float64(1.0690733733970572e-05),
  'shift_y': np.float64(-1.8707688902461266e-05),
  'rot_s_rad': np.float64(-0.00020222885353878464)},
 'bpm_qd14f': {'shift_x': np.float64(-2.8842625296327298e-06),
  'shift_y': np.float64(5.6793081746057836e-05),
  'rot_s_rad': np.float64(-6.704773286363154e-05)},
 'bpm_qf15f': {'shift_x': np.float64(-8.93487587851483e-05),
  'shift_y': np.float64(0.00022654309876329052),
  'rot_s_rad': np.float64(-0.00011438387078656145)},
 'bpm_qd16f': {'shift_x': np.float64(2.219179540058857e-05),
  'shift_y': np.float64(0.00011108596447668471),
  'rot_s_rad': np.float64(9.67683655009108e-05)},
 'bpm_qf17f': {'shift_x': np.float64(-0.0001420434611887312),
  'shift_y': np.float64(0.0001556552481676292),
  'rot_s_rad': np.float64(9.28688616196329e-05)},
 'bpm_qd18f': {'shift_x': 

In [26]:
# Set switches for powering the sextupoles and deactivate them

# Arc: ['s[df][12]a', 'sf3m', 'sf4i', 's[df][12]bf', 's[fd][123][fdci]', 'sf4[cdf]']
create_elements_switch(line, pattern=['s[df][12]a', 'sf3m', 'sf4i', 's[df][12]bf', 's[fd][123][fdci]', 'sf4[cdf]'], switch_name='on_powering_sext_arc', element_type = 'Sextupole')
line.vars['on_powering_sext_arc'] = 0

# IR sextupoles(excluding crab): s[fd][mxy][12][lr]
create_elements_switch(line, pattern='s[fd][mxy][12][lr]', 
                    switch_name='on_powering_sext_IR', element_type = 'Sextupole')
line.vars['on_powering_sext_IR'] = 0

# Crab sextupoles: scrab
create_elements_switch(line, pattern='scrab', switch_name='on_powering_sext_crab', element_type = 'Sextupole')
line.vars['on_powering_sext_crab'] = 0

print('Switches for powering the sextupoles created successfully!')

Switches for powering the sextupoles created successfully!


In [27]:
# Load the steering correctors and monitors required for the correction procedure
steering_monitors=tt.rows['bpm.*']
line.steering_monitors_x = steering_monitors.name
line.steering_monitors_y = steering_monitors.name
print(f'Found', len(line.steering_monitors_x), 'BPMs in the line.')

line.steering_correctors_x = tt.rows['hcor.*'].name
line.steering_correctors_y = tt.rows['vcor.*'].name
print(f'Found', len(line.steering_correctors_x), 'horizontal correctors in the line.')
print(f'Found', len(line.steering_correctors_y), 'vertical correctors in the line.')

Found 2219 BPMs in the line.
Found 2259 horizontal correctors in the line.
Found 2259 vertical correctors in the line.


In [28]:
# Set chromatic properties to False
line.twiss_default['compute_chromatic_properties'] = False
print('Switched to compute_chromatic_properties=False.')

Switched to compute_chromatic_properties=False.


### Enabling the imperfections

In [29]:
# If needed, set some of the sextupole powering switches to a non-zero value for the first orbit corrections
line.vars['on_powering_sext_arc'] = 0
line.vars['on_powering_sext_IR'] = 0
line.vars['on_powering_sext_crab'] = 0

# Turn on the switches for the misalignments of the arc dipoles and arc quads
line.vars['on_misalignment_dip_arc'] = 1
line.vars['on_misalignment_quad_arc'] = 1
line.vars['on_misalignment_sext_arc'] = 0 # Note that sextupoles must be powered to turn on the misalignments!
line.vars['on_misalignment_girder'] = 0 # To be ramped up gradually!

line.vars['on_misalignment_dip_ip'] = 1
line.vars['on_misalignment_dip_non_ip'] = 1
line.vars['on_misalignment_quad_fd'] = 1 # Careful with the FD quads, they are very strong!
line.vars['on_misalignment_quad_fd_cs'] = 1
line.vars['on_misalignment_quad_cs_arc'] = 1
line.vars['on_misalignment_sext_fd_cs'] = 0 # Note that this depends on the powering switches for sext_IR and sext_crab to be enabled!

# Turn on the switches for the field errors of the arc dipoles and arc quads
line.vars['on_field_error_dip_arc'] = 1
line.vars['on_field_error_quad_arc'] = 1
line.vars['on_field_error_sext_arc'] = 0 # Note that sextupoles must be powered to turn on the field errors!

line.vars['on_field_error_dip_ip'] = 1
line.vars['on_field_error_dip_non_ip'] = 1
line.vars['on_field_error_quad_fd'] = 1 # Careful with the FD quads, they are very strong!
line.vars['on_field_error_quad_fd_cs'] = 1
line.vars['on_field_error_quad_cs_arc'] = 1
line.vars['on_field_error_sext_fd_cs'] = 0 # Note that this depends on the powering switches for sext_IR and sext_crab to be enabled!


### Performing the orbit correction

In [30]:
# CHOOSE NUMBER OF SINGULAR VALUES FOR ORBIT CORRECTION HERE
num_sing_vals = 2200

In [31]:
# First orbit correction with deactivated sextupoles

apply_orbit_correction(line=line, twiss_table=tw_ref, num_sing_vals=num_sing_vals)
print('First orbit correction with deactivated sextupoles was successful!')

match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
print('Tune matched successfully after first orbit correction with deactivated sextupoles!')



Warning! Need second attempt on closed orbit search


Starting orbit correction with threading method...
Stop at s=5000, global rms = [x: 9.24e-04 -> 1.45e-06, y: 1.66e-03 -> 5.33e-06]
Stop at s=10000, global rms = [x: 7.86e-04 -> 1.93e-06, y: 1.03e-03 -> 1.85e-06]
Stop at s=15000, global rms = [x: 3.01e-03 -> 2.35e-05, y: 2.95e-03 -> 6.95e-05]
Stop at s=20000, global rms = [x: 3.69e-04 -> 8.74e-07, y: 1.23e-03 -> 2.32e-06]
Stop at s=25000, global rms = [x: 5.50e-04 -> 2.27e-06, y: 4.12e-04 -> 1.27e-06]
Stop at s=30000, global rms = [x: 5.76e-04 -> 6.78e-07, y: 8.06e-04 -> 9.23e-07]
Stop at s=35000, global rms = [x: 9.07e-04 -> 4.97e-06, y: 6.87e-04 -> 4.92e-06]
Stop at s=40000, global rms = [x: 5.43e-04 -> 6.23e-07, y: 1.01e-03 -> 4.73e-06]
Stop at s=45000, global rms = [x: 2.04e-04 -> 1.04e-06, y: 3.14e-04 -> 1.55e-06]
Stop at s=50000, global rms = [x: 2.95e-04 -> 4.03e-07, y: 2.83e-04 -> 5.59e-07]
Stop at s=55000, global rms = [x: 3.64e-04 -> 5.00e-07, y: 3.74e-04 -> 1.26e-06]
Sto

In [32]:
# Girder ramping
    
for i in [0.25, 0.50, 0.75, 1]:
    line.vars['on_misalignment_girder'] = i
    
    apply_orbit_correction(line, tw_ref, num_sing_vals=num_sing_vals)
    print(f'Orbit corrected successfully for girder misalignment of {i*100}% !')
    
    match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
    print(f'Tune matched successfully after orbit correction with girder misalignment of {i*100}% !')

Iteration 0, x_rms: 1.37e-03 -> 2.64e-05, y_rms: 1.69e-03 -> 2.49e-04
Iteration 1, x_rms: 2.64e-05 -> 2.74e-06, y_rms: 2.49e-04 -> 1.75e-05
Iteration 2, x_rms: 2.74e-06 -> 1.17e-07, y_rms: 1.75e-05 -> 2.77e-06
Iteration 3, x_rms: 1.17e-07 -> 2.77e-08, y_rms: 2.77e-06 -> 1.93e-07
Iteration 4, x_rms: 2.77e-08 -> 1.25e-09, y_rms: 1.93e-07 -> 3.16e-08
Iteration 5, x_rms: 1.25e-09 -> 3.16e-10, y_rms: 3.16e-08 -> 2.13e-09
Iteration 6, x_rms: 3.16e-10 -> 1.36e-11, y_rms: 2.13e-09 -> 3.61e-10
Iteration 7, x_rms: 1.36e-11 -> 3.59e-12, y_rms: 3.61e-10 -> 2.35e-11
Orbit corrected successfully for girder misalignment of 25.0% !
                                             
Optimize - start penalty: 0.0006792                         
Matching: model call n. 6 penalty = 2.5987e-08              
Optimize - end penalty:  2.59867e-08                            
Target status:               alty = 2.5987e-08              
id state tag  tol_met       residue   current_val    target_val description       

In [33]:
# Arc sext ramping
    
line.vars['on_misalignment_sext_arc'] = 1
line.vars['on_field_error_sext_arc'] = 1

for i in [0.33, 0.66, 0.90, 1]:
    print(f'Ramping up arc sextupoles to {i*100}%...')
    line.vars['on_powering_sext_arc'] = i

    apply_orbit_correction(line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals) # include the monitor misalignments in the orbit correction procedure now
    print(f'Orbit corrected successfully for arc sextupole ramping up to {i*100}% !')

    match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
    print(f'Tune matched successfully after orbit correction with arc sextupole ramping up to {i*100}% !')

print('All orbit corrections (incl. tune matching) for arc sextupole ramping were completed successfully!')

Ramping up arc sextupoles to 33.0%...
Iteration 0, x_rms: 1.65e-04 -> 5.78e-06, y_rms: 1.33e-04 -> 7.24e-06
Iteration 1, x_rms: 5.78e-06 -> 4.33e-06, y_rms: 7.24e-06 -> 6.39e-06
Iteration 2, x_rms: 4.33e-06 -> 4.33e-06, y_rms: 6.39e-06 -> 6.38e-06
Orbit corrected successfully for arc sextupole ramping up to 33.0% !
                                             
Optimize - start penalty: 0.009575                          
Matching: model call n. 6 penalty = 1.0092e-05              
Optimize - end penalty:  1.0092e-05                            
Target status:               alty = 1.0092e-05              
id state tag  tol_met       residue   current_val    target_val description                            
0  ON    tune    True  -7.08282e-07       194.167       194.167 'qx', val=194.167, tol=1e-05, weight=10
1  ON    tune    True   7.18904e-07         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10  
Vary status:                 
id state tag  met name lower_limit   current_val

In [34]:
# IR and crab sext ramping
line.vars['on_misalignment_sext_fd_cs'] = 1
line.vars['on_field_error_sext_fd_cs'] = 1

for i in [0.33, 0.66, 0.90, 1]:
    print(f'Ramping up IR and crab sextupoles to {i*100}%...')
    line.vars['on_powering_sext_IR'] = i
    line.vars['on_powering_sext_crab'] = i

    apply_orbit_correction(line, tw_ref, monitor_alignment, num_sing_vals=num_sing_vals)
    print(f'Orbit corrected successfully for IR and crab sextupole ramping up to {i*100}% !')
    
    match_tune_chroma(line, tw_ref, match_quantities='tune', method='4d')
    print(f'Tune matched successfully after orbit correction with IR and crab sextupole ramping up to {i*100}% !')

print('All orbit corrections (incl. tune matching) for IR and crab sextupole ramping were completed successfully!')

print('The full orbit correction (incl. tune matching) was completed successfully!')

Ramping up IR and crab sextupoles to 33.0%...
Iteration 0, x_rms: 4.33e-06 -> 4.33e-06, y_rms: 6.56e-06 -> 6.45e-06
Orbit corrected successfully for IR and crab sextupole ramping up to 33.0% !
                                             
Optimize - start penalty: 0.4695                            
Matching: model call n. 16 penalty = 3.2975e-07              
Optimize - end penalty:  3.29751e-07                            
Target status:               nalty = 3.2975e-07              
id state tag  tol_met       residue   current_val    target_val description                            
0  ON    tune    True   2.36494e-08       194.167       194.167 'qx', val=194.167, tol=1e-05, weight=10
1  ON    tune    True  -2.29796e-08         170.2         170.2 'qy', val=170.2, tol=1e-05, weight=10  
Vary status:                 
id state tag  met name lower_limit   current_val upper_limit val_at_iter_0          step        weight
0  ON    quad OK  kqf2 None           0.00927003 None           0.

In [35]:
# Reintroduce the chromatic properties
line.twiss_default['compute_chromatic_properties'] = True
print('Switched to compute_chromatic_properties=True since all sextupoles are now fully ramped up!')

Switched to compute_chromatic_properties=True since all sextupoles are now fully ramped up!


In [36]:
# Save the orbit-corrected line
line.to_json(f'{line_version}_line_01_orbit_corrected_tune_matched_seed{seed}.json')
# The saved line is cycled to rf cavity, has radiation disabled, and uses the 4D Twiss method.

print(f'Finished the process for seed {seed}!')

Finished the process for seed 4!
